In [ ]:
# ============================================================
# CELL 1: Install Dependencies — PINNED EXACT VERSIONS
# ============================================================
# Same pinned stack as Project 1. No PEFT/TRL needed here because
# we are ONLY doing inference — no fine-tuning, no adapters.
#
# transformers 5.0.0 is pinned because kaggle-environments depends
# on it (downgrading would break Kaggle's own environment — same
# reason as Project 1 Appendix A).
#
# bitsandbytes 0.49.2 ships CUDA 12.8 pre-built binaries that work
# with Kaggle's CUDA 12.8 runtime.
# ============================================================

!pip install -q \
    "transformers==5.0.0" \
    "bitsandbytes==0.49.2" \
    "accelerate>=0.29.0" \
    "huggingface_hub" \
    tqdm \
    pandas \
    matplotlib \
    seaborn \
    tabulate

print("✅ All packages installed.")

# Verify exact versions
import transformers, bitsandbytes, torch
print(f"transformers:  {transformers.__version__}")
print(f"bitsandbytes:  {bitsandbytes.__version__}")
print(f"torch:         {torch.__version__}")
print(f"CUDA:          {torch.version.cuda}")

In [ ]:
# ============================================================
# DAY 4 — CELL 2: GPU Verification & Global Imports
# ============================================================
# Verify both T4s are visible. If only one shows, recheck
# Settings → Accelerator → GPU T4 x2.
# ============================================================
import os
import gc
import time
import json
import statistics
from datetime import datetime
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# Expandable segments reduces memory fragmentation — same setting
# used in Project 1 Day 2 Cell 5. Costs nothing.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"CUDA available:    {torch.cuda.is_available()}")
print(f"GPU count:         {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | "
          f"{props.total_memory/1e9:.1f} GB | "
          f"CC {props.major}.{props.minor}")

# Expected: 2x NVIDIA T4, 15.8 GB each, compute capability 7.5
assert torch.cuda.device_count() >= 1, "❌ No GPU detected"
assert torch.cuda.is_available(),       "❌ CUDA not available"
print("\n✅ GPU verified. Proceed to Cell 3.")

In [ ]:
# ============================================================
# DAY 4 — CELL 3: Experiment Matrix
# ============================================================
# Define the full experiment grid ONCE, as module-level constants.
# Every downstream cell reads from these — if we want to change
# the grid, we change it here and only here.
#
# Grid size reasoning:
#   3 quant levels × 4 batch × 4 seqlen × 10 reps = 480 generate() calls
#   Avg call ≈ 1-3s on T4 depending on config
#   Total compute ≈ 1.5-3 hours across all 3 quant sweeps
#   Fits comfortably in Kaggle's 12hr session limit.
#
# max_new_tokens = 50:
#   Matches realistic short-answer workloads (chat completions,
#   SQL queries, classification). Longer max_new_tokens would
#   amortize prefill cost and overstate decode throughput.
# ============================================================

MODEL_NAME         = "microsoft/phi-2"
QUANT_LEVELS       = ["fp16", "int8", "int4"]
BATCH_SIZES        = [1, 4, 8, 16]
SEQ_LENS           = [64, 128, 256, 512]
MAX_NEW_TOKENS     = 50
N_RUNS_PER_CONFIG  = 10
N_WARMUP_RUNS      = 2       # CUDA kernel compile + KV cache init
SEED               = 42

# Output directory
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "charts").mkdir(exist_ok=True)

# Hardware metadata — logged with every result row so numbers
# are never unmoored from the hardware they came from
HARDWARE_META = {
    "gpu_name":       torch.cuda.get_device_name(0),
    "gpu_count":      torch.cuda.device_count(),
    "cuda_version":   torch.version.cuda,
    "torch_version":  torch.__version__,
    "transformers":   __import__("transformers").__version__,
    "bitsandbytes":   __import__("bitsandbytes").__version__,
    "python":         __import__("sys").version.split()[0],
}

# Set global seed — ensures dummy input tokens are identical
# across quant levels, so comparisons are apples-to-apples
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Experiment matrix:")
print(f"  Model:            {MODEL_NAME}")
print(f"  Quant levels:     {QUANT_LEVELS}")
print(f"  Batch sizes:      {BATCH_SIZES}")
print(f"  Sequence lens:    {SEQ_LENS}")
print(f"  Max new tokens:   {MAX_NEW_TOKENS}")
print(f"  Runs per config:  {N_RUNS_PER_CONFIG}")
print(f"  Warmup runs:      {N_WARMUP_RUNS}")
print(f"  Total generates:  "
      f"{len(QUANT_LEVELS) * len(BATCH_SIZES) * len(SEQ_LENS) * (N_RUNS_PER_CONFIG + N_WARMUP_RUNS)}")
print(f"\nHardware:")
for k, v in HARDWARE_META.items():
    print(f"  {k:<14} {v}")

In [ ]:
# ============================================================
# DAY 4 — CELL 4: LLMBenchmarker — Core Profiler
# ============================================================
# DESIGN NOTES (read before modifying this cell)
#
# 1. Quantization loading must use `quantization_config=BitsAndBytesConfig(...)`.
#    In transformers 5.0.0, `load_in_8bit=True` and `load_in_4bit=True` as
#    direct `from_pretrained` kwargs are DEPRECATED. The Project 2 master
#    doc (in anthropic_fellows_master_context.md) had it wrong — fixed here.
#
# 2. For FP16 we pass `dtype=torch.float16` (v5.0.0 replaces `torch_dtype`
#    with `dtype`; old name still works but emits DeprecationWarning).
#
# 3. Phi-2 needs the pad_token_id config patch — same bug as Project 1
#    Appendix A. PhiConfig in transformers 5.0.0 is missing the
#    pad_token_id attribute entirely; PhiModel.__init__ reads it at
#    line 348 and AttributeErrors. Fix: inject via config.__dict__.
#
# 4. Timing discipline (this is the part that matters most):
#    - `torch.cuda.synchronize()` BEFORE start timer — flush any queued
#      kernels from prior ops
#    - `torch.cuda.synchronize()` AFTER end timer — wait for GPU to
#      finish (`generate()` returns before all kernels complete)
#    - Without both synchronize() calls, measurements are random noise.
#
# 5. Warmup runs are discarded. First `generate()` after load is always
#    slower because CUDA kernels JIT-compile on first use and KV cache
#    allocators warm up. 2 warmup runs is plenty.
#
# 6. Peak VRAM is captured via `torch.cuda.max_memory_allocated(i)` summed
#    across GPUs, and reset with `reset_peak_memory_stats()` before each
#    config. Otherwise peak stays at the maximum seen across the whole
#    sweep — useless for per-config reporting.
#
# 7. TTFT (time-to-first-token) is measured with a separate single-token
#    generate pass. TTFT ≈ prefill time, which is bandwidth-bound for
#    long prompts and differs from per-token decode cost.
# ============================================================

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    BitsAndBytesConfig,
)


def _load_tokenizer_and_patched_config(model_name: str):
    """
    Load tokenizer + AutoConfig, patch PhiConfig.pad_token_id.
    Same pattern as Project 1 Day 2 Cell 2.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    config.__dict__["pad_token_id"] = tokenizer.pad_token_id

    return tokenizer, config


def load_model(model_name: str, quantization: str):
    """
    Load Phi-2 at the requested precision.
    quantization ∈ {"fp16", "int8", "int4"}.

    Returns (model, tokenizer).
    """
    tokenizer, config = _load_tokenizer_and_patched_config(model_name)

    common_kwargs = dict(
        config              = config,
        device_map          = "auto",          # distribute across both T4s
        trust_remote_code   = True,
    )

    if quantization == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype = torch.float16,              # v5.0.0 name (was torch_dtype)
            **common_kwargs,
        )

    elif quantization == "int8":
        # LLM.int8() — requires compute capability 7.5+ (T4 is 7.5 ✓)
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config = bnb_cfg,
            **common_kwargs,
        )

    elif quantization == "int4":
        # NF4 with double quantization — requires CC 6.0+ (T4 is 7.5 ✓)
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.float16,
            bnb_4bit_use_double_quant = True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config = bnb_cfg,
            **common_kwargs,
        )
    else:
        raise ValueError(f"Unknown quantization: {quantization}")

    model.eval()
    return model, tokenizer


def _sum_peak_vram_gb() -> float:
    """Peak VRAM across all visible GPUs, in GB."""
    return sum(
        torch.cuda.max_memory_allocated(i)
        for i in range(torch.cuda.device_count())
    ) / 1e9


def _reset_peak_vram():
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)


def make_dummy_inputs(tokenizer, batch_size: int, seq_len: int, device):
    """
    Generate deterministic dummy input tokens. Same seed = same inputs
    across all quant levels, so results are directly comparable.

    We avoid token IDs 0 and tokenizer.eos_token_id to prevent early
    stop / padding-only sequences.
    """
    g = torch.Generator(device="cpu").manual_seed(SEED + batch_size * 100 + seq_len)
    vocab_lo, vocab_hi = 100, min(tokenizer.vocab_size - 1, 50000)
    ids = torch.randint(vocab_lo, vocab_hi, (batch_size, seq_len), generator=g)
    attn = torch.ones_like(ids)
    return ids.to(device), attn.to(device)


def benchmark_config(
    model,
    tokenizer,
    batch_size: int,
    seq_len: int,
    max_new_tokens: int = MAX_NEW_TOKENS,
    n_runs: int = N_RUNS_PER_CONFIG,
    n_warmup: int = N_WARMUP_RUNS,
) -> dict:
    """
    Benchmark ONE (batch, seq_len) configuration.
    Returns a dict of measured metrics.
    Raises torch.cuda.OutOfMemoryError on OOM — caller should catch.
    """
    device = next(model.parameters()).device
    input_ids, attn_mask = make_dummy_inputs(tokenizer, batch_size, seq_len, device)

    gen_kwargs = dict(
        attention_mask        = attn_mask,
        do_sample             = False,             # deterministic/greedy
        pad_token_id          = tokenizer.eos_token_id,
        eos_token_id          = None,              # don't stop early — measure full decode
        return_dict_in_generate = True,
    )

    # ── Warmup (not measured) ─────────────────────────────────────────
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model.generate(
                input_ids,
                max_new_tokens = max_new_tokens,
                **gen_kwargs,
            )
    torch.cuda.synchronize()

    # ── TTFT measurement: single-token generate ───────────────────────
    # Prefill-dominated time — bandwidth-bound on long prompts.
    _reset_peak_vram()
    ttft_times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model.generate(
                input_ids,
                max_new_tokens = 1,
                **gen_kwargs,
            )
        torch.cuda.synchronize()
        ttft_times.append(time.perf_counter() - t0)

    # ── Full generate: prefill + N decode steps ───────────────────────
    _reset_peak_vram()
    wall_times = []
    tokens_per_seq = None
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens = max_new_tokens,
                **gen_kwargs,
            )
        torch.cuda.synchronize()
        wall_times.append(time.perf_counter() - t0)
        tokens_per_seq = output.sequences.shape[1] - seq_len   # actual new tokens (per seq)

    peak_vram_gb = _sum_peak_vram_gb()

    # ── Derived metrics ───────────────────────────────────────────────
    total_tokens = n_runs * batch_size * tokens_per_seq
    total_time   = sum(wall_times)
    throughput_tps = total_tokens / total_time                 # tokens/sec (aggregate)

    mean_wall = statistics.mean(wall_times)
    # Per-sequence token latency: how long one token takes for one sequence in the batch
    latency_ms_per_tok = (mean_wall / tokens_per_seq) * 1000

    # Decode-only latency (subtract TTFT to isolate steady-state decode)
    mean_ttft = statistics.mean(ttft_times)
    decode_only_time = mean_wall - mean_ttft
    decode_ms_per_tok = (decode_only_time / max(tokens_per_seq - 1, 1)) * 1000

    return {
        "batch_size":             batch_size,
        "seq_len":                seq_len,
        "max_new_tokens":         max_new_tokens,
        "tokens_per_seq":         tokens_per_seq,
        "throughput_tps":         round(throughput_tps, 2),
        "latency_ms_per_tok":     round(latency_ms_per_tok, 3),
        "decode_ms_per_tok":      round(decode_ms_per_tok, 3),
        "ttft_ms":                round(mean_ttft * 1000, 2),
        "wall_time_std_s":        round(statistics.stdev(wall_times), 4) if n_runs > 1 else 0.0,
        "peak_vram_gb":           round(peak_vram_gb, 3),
        "n_runs":                 n_runs,
        "error":                  None,
    }


def run_sweep(
    model,
    tokenizer,
    quant: str,
    batch_sizes,
    seq_lens,
) -> pd.DataFrame:
    """Run the full (batch × seq_len) grid for ONE quantization level."""
    rows = []
    configs = [(b, s) for b in batch_sizes for s in seq_lens]

    pbar = tqdm(configs, desc=f"{quant:>5}")
    for batch_size, seq_len in pbar:
        try:
            row = benchmark_config(model, tokenizer, batch_size, seq_len)
            row["status"] = "ok"
            pbar.set_postfix({
                "bs": batch_size, "sl": seq_len,
                "tps": f"{row['throughput_tps']:.0f}",
                "vram": f"{row['peak_vram_gb']:.1f}GB",
            })
        except torch.cuda.OutOfMemoryError:
            row = {
                "batch_size": batch_size, "seq_len": seq_len,
                "max_new_tokens": MAX_NEW_TOKENS,
                "tokens_per_seq": None,
                "throughput_tps": None, "latency_ms_per_tok": None,
                "decode_ms_per_tok": None, "ttft_ms": None,
                "wall_time_std_s": None, "peak_vram_gb": None,
                "n_runs": 0, "error": "OOM", "status": "oom",
            }
            # Clear cache before next config — OOM leaves partial allocations
            torch.cuda.empty_cache()
            pbar.set_postfix({"bs": batch_size, "sl": seq_len, "status": "OOM"})

        row["quantization"] = quant
        row["model"]        = MODEL_NAME
        row["timestamp"]    = datetime.now().isoformat()
        for k, v in HARDWARE_META.items():
            row[f"hw_{k}"] = v
        rows.append(row)

    return pd.DataFrame(rows)


print("✅ LLMBenchmarker functions defined. Proceed to Cell 5.")

In [ ]:
# ============================================================
# DAY 4 — CELL 5: FP16 Baseline Sweep
# ============================================================
# FP16 is our reference point. All speedup and memory-reduction
# claims are measured relative to this baseline.
#
# Expected runtime: ~20-40 minutes (16 configs × ~1-2 min each
# including warmup). Watch the tqdm postfix for throughput and
# VRAM — if you see the numbers scaling reasonably with batch
# size, the measurement loop is working.
# ============================================================

print("Loading Phi-2 in FP16 across both T4s...")
model_fp16, tokenizer = load_model(MODEL_NAME, "fp16")
print(f"✅ Loaded.")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

print("\nRunning FP16 sweep...")
df_fp16 = run_sweep(model_fp16, tokenizer, "fp16", BATCH_SIZES, SEQ_LENS)

# Save immediately — if session dies, we haven't lost FP16 data
fp16_csv = RESULTS_DIR / "bench_fp16.csv"
df_fp16.to_csv(fp16_csv, index=False)
print(f"\n✅ FP16 sweep complete. Saved to {fp16_csv}")
print(df_fp16[[
    "batch_size", "seq_len", "throughput_tps",
    "latency_ms_per_tok", "ttft_ms", "peak_vram_gb", "status"
]].to_string(index=False))

In [ ]:
# ============================================================
# DAY 4 — CELL 6: Memory Cleanup
# ============================================================
# Drop the FP16 model BEFORE loading INT8 on Day 5. Without this,
# we keep ~5GB of weights sitting on GPU that we don't need.
# ============================================================
import gc

try:
    del model_fp16
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# Reset peak stats so tomorrow's INT8 numbers start from zero
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved  = torch.cuda.memory_reserved(i) / 1e9
    print(f"GPU {i}: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

print("\n✅ Memory cleared. Day 4 complete.")

In [ ]:
# ============================================================
# DAY 4 — CELL 7: Day 4 Checkpoint
# ============================================================
import os

print("=" * 60)
print("DAY 4 COMPLETE — CHECKPOINT SUMMARY")
print("=" * 60)

checks = {
    "results/bench_fp16.csv":  os.path.exists("results/bench_fp16.csv"),
}
for path, exists in checks.items():
    print(f"   {'✅' if exists else '❌'} {path}")

# Quick sanity — did FP16 produce finite numbers?
df = pd.read_csv("results/bench_fp16.csv")
ok_rows = df[df["status"] == "ok"]
print(f"\n   Successful configs:  {len(ok_rows)} / {len(df)}")
print(f"   Throughput range:    "
      f"{ok_rows['throughput_tps'].min():.0f} - "
      f"{ok_rows['throughput_tps'].max():.0f} tok/s")
print(f"   Peak VRAM range:     "
      f"{ok_rows['peak_vram_gb'].min():.2f} - "
      f"{ok_rows['peak_vram_gb'].max():.2f} GB")

print("\n💾 Save Version → Save & Run All before closing (captures CSV).")
print("✅ Ready to proceed to Day 5.")
print("=" * 60)

In [ ]:
# ============================================================
# DAY 5 — CELL 1: Reload Environment
# ============================================================
# If this is a fresh Kaggle session, re-run Day 4 Cells 1, 2, 3, 4.
# If continuing the same session, this cell is a no-op safety net.
# ============================================================
import os
if "run_sweep" not in dir():
    raise RuntimeError(
        "❌ LLMBenchmarker functions not defined. "
        "Re-run Day 4 Cells 1, 2, 3, 4 before proceeding."
    )
print("✅ Environment ready.")

In [ ]:
# ============================================================
# DAY 5 — CELL 2: INT8 Sweep (LLM.int8())
# ============================================================
# LLM.int8() uses mixed-precision decomposition — it keeps outlier
# features in FP16 and quantizes the rest to INT8. This typically
# has negligible quality impact.
#
# Expected memory: ~2.7 GB for Phi-2 weights (half of FP16's 5.4 GB).
# Expected throughput: often LOWER than FP16 on T4, because the
# outlier-detection path adds overhead. This is a real finding, not
# a bug — document it in the analysis.
# ============================================================
print("Loading Phi-2 in INT8...")
model_int8, tokenizer = load_model(MODEL_NAME, "int8")
print(f"✅ Loaded.")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

print("\nRunning INT8 sweep...")
df_int8 = run_sweep(model_int8, tokenizer, "int8", BATCH_SIZES, SEQ_LENS)

int8_csv = RESULTS_DIR / "bench_int8.csv"
df_int8.to_csv(int8_csv, index=False)
print(f"\n✅ INT8 sweep complete. Saved to {int8_csv}")
print(df_int8[[
    "batch_size", "seq_len", "throughput_tps",
    "latency_ms_per_tok", "ttft_ms", "peak_vram_gb", "status"
]].to_string(index=False))

In [ ]:
# ============================================================
# DAY 5 — CELL 3: Cleanup Between Quant Levels
# ============================================================
import gc

try:
    del model_int8
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)
    allocated = torch.cuda.memory_allocated(i) / 1e9
    print(f"GPU {i} allocated after cleanup: {allocated:.2f} GB")

print("\n✅ Ready for INT4 sweep.")

In [ ]:
# ============================================================
# DAY 5 — CELL 4: INT4 Sweep (NF4 + double quantization)
# ============================================================
# NF4 (NormalFloat4) is information-theoretically optimal for
# normally-distributed weights — it packs more information per
# bit than uniform INT4.
#
# Double quantization = quantize the quantization constants
# themselves. Saves an additional ~0.4 bits/param on average.
#
# Expected memory: ~1.4 GB for Phi-2 weights. Combined with
# KV cache growth at large batch/seqlen, this is where we
# start seeing the VRAM benefit.
# ============================================================
print("Loading Phi-2 in INT4 (NF4 + double quant)...")
model_int4, tokenizer = load_model(MODEL_NAME, "int4")
print(f"✅ Loaded.")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

print("\nRunning INT4 sweep...")
df_int4 = run_sweep(model_int4, tokenizer, "int4", BATCH_SIZES, SEQ_LENS)

int4_csv = RESULTS_DIR / "bench_int4.csv"
df_int4.to_csv(int4_csv, index=False)
print(f"\n✅ INT4 sweep complete. Saved to {int4_csv}")
print(df_int4[[
    "batch_size", "seq_len", "throughput_tps",
    "latency_ms_per_tok", "ttft_ms", "peak_vram_gb", "status"
]].to_string(index=False))

In [ ]:
# ============================================================
# DAY 5 — CELL 5: Combine + Save Master CSV
# ============================================================
# Single source of truth for Day 6 analysis.
# ============================================================
import pandas as pd

df_fp16_reload = pd.read_csv(RESULTS_DIR / "bench_fp16.csv")
df_int8_reload = pd.read_csv(RESULTS_DIR / "bench_int8.csv")
df_int4_reload = pd.read_csv(RESULTS_DIR / "bench_int4.csv")

df_all = pd.concat(
    [df_fp16_reload, df_int8_reload, df_int4_reload],
    ignore_index=True,
)

master_csv = RESULTS_DIR / "bench_all.csv"
df_all.to_csv(master_csv, index=False)

print(f"✅ Master results saved to {master_csv}")
print(f"   Total rows:           {len(df_all)}")
print(f"   Successful rows:      {(df_all['status']=='ok').sum()}")
print(f"   OOM rows:             {(df_all['status']=='oom').sum()}")
print(f"\n   By quantization:")
print(df_all.groupby("quantization")["status"].value_counts())

In [ ]:
# ============================================================
# DAY 5 — CELL 6: Cleanup + Day 5 Checkpoint
# ============================================================
import gc, os

try:
    del model_int4
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

required = [
    "results/bench_fp16.csv",
    "results/bench_int8.csv",
    "results/bench_int4.csv",
    "results/bench_all.csv",
]
print("=" * 60)
print("DAY 5 CHECKPOINT")
print("=" * 60)
for f in required:
    print(f"   {'✅' if os.path.exists(f) else '❌'} {f}")

print("\n💾 Save Version → Save & Run All before closing.")
print("✅ Ready to proceed to Day 6.")
print("=" * 60)

In [ ]:
# ============================================================
# DAY 6 — CELL 1: Load Master CSV, Compute Derived Metrics
# ============================================================
import pandas as pd
import numpy as np

df = pd.read_csv("results/bench_all.csv")
print(f"Loaded {len(df)} rows.")

# ── Derived metric: speedup vs FP16 baseline at same (bs, sl) ─────
baseline = (
    df[df["quantization"] == "fp16"]
    [["batch_size", "seq_len", "throughput_tps", "peak_vram_gb"]]
    .rename(columns={
        "throughput_tps": "fp16_throughput",
        "peak_vram_gb":   "fp16_peak_vram_gb",
    })
)
df = df.merge(baseline, on=["batch_size", "seq_len"], how="left")

df["speedup_vs_fp16"]       = df["throughput_tps"] / df["fp16_throughput"]
df["vram_reduction_vs_fp16"] = 1 - (df["peak_vram_gb"] / df["fp16_peak_vram_gb"])

# ── Derived metric: efficiency (tok/s per GB of VRAM) ─────────────
# Useful for "if I had to pick one number" decisions
df["tokens_per_sec_per_gb"] = df["throughput_tps"] / df["peak_vram_gb"]

df.to_csv("results/bench_all_enriched.csv", index=False)
print("✅ Enriched CSV saved.")

# Show the core comparison at batch=1, seq_len=128 — standard single-user config
core = df[
    (df["batch_size"] == 1) &
    (df["seq_len"]    == 128) &
    (df["status"]     == "ok")
].sort_values("quantization")

print("\nCore comparison (batch=1, seq_len=128):")
print(core[[
    "quantization", "throughput_tps", "latency_ms_per_tok",
    "ttft_ms", "peak_vram_gb", "speedup_vs_fp16",
]].to_string(index=False))

In [ ]:
# ============================================================
# DAY 6 — CELL 2: Publication-Ready Summary Table
# ============================================================
# Report numbers at a single standard config for readability.
# Full grid goes in results/bench_all.csv.
# ============================================================
import pandas as pd

REPRESENTATIVE_BATCH = 1
REPRESENTATIVE_SL    = 128

rep = df[
    (df["batch_size"] == REPRESENTATIVE_BATCH) &
    (df["seq_len"]    == REPRESENTATIVE_SL) &
    (df["status"]     == "ok")
].copy()

summary_rows = []
for _, r in rep.sort_values("quantization").iterrows():
    summary_rows.append({
        "Quantization":      r["quantization"].upper(),
        "Throughput (tok/s)": f"{r['throughput_tps']:.1f}",
        "Latency (ms/tok)":   f"{r['latency_ms_per_tok']:.2f}",
        "TTFT (ms)":          f"{r['ttft_ms']:.1f}",
        "Peak VRAM (GB)":     f"{r['peak_vram_gb']:.2f}",
        "vs FP16 speedup":    f"{r['speedup_vs_fp16']:.2f}×",
        "VRAM reduction":     (
            f"{r['vram_reduction_vs_fp16']*100:.0f}%"
            if r["quantization"] != "fp16" else "—"
        ),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results/results_summary.csv", index=False)

print("=" * 70)
print(f"RESULTS SUMMARY (batch={REPRESENTATIVE_BATCH}, seq_len={REPRESENTATIVE_SL})")
print("=" * 70)
print(summary_df.to_markdown(index=False))

In [ ]:
# ============================================================
# DAY 6 — CELL 3: Output Quality Validation
# ============================================================
# A fast model that generates garbage is useless. Before claiming
# INT4 is a "2.8x speedup", we should confirm the model still
# produces sensible output at each quant level on realistic prompts.
#
# We run the same 5 prompts through each quant level, look at the
# output side-by-side. This isn't a rigorous quality eval — it's a
# sanity check.
# ============================================================

VALIDATION_PROMPTS = [
    "The three primary colors are",
    "In Python, to reverse a list you can use",
    "Write a SQL query to find the oldest employee in a table called staff:",
    "The main difference between TCP and UDP is",
    "Explain in one sentence what a binary search tree is:",
]

def load_and_generate(quant: str, prompts, max_new_tokens=60):
    print(f"\n── Loading {quant.upper()} ──")
    mdl, tok = load_model(MODEL_NAME, quant)
    mdl.eval()
    out = []
    for p in prompts:
        ids = tok(p, return_tensors="pt").to(next(mdl.parameters()).device)
        with torch.no_grad():
            gen = mdl.generate(
                **ids,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                pad_token_id   = tok.eos_token_id,
            )
        text = tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        # Take first line only for readability
        out.append(text.split("\n")[0].strip())
    # Cleanup
    del mdl
    import gc; gc.collect(); torch.cuda.empty_cache()
    return out

# Run each quant level once
outputs_fp16 = load_and_generate("fp16", VALIDATION_PROMPTS)
outputs_int8 = load_and_generate("int8", VALIDATION_PROMPTS)
outputs_int4 = load_and_generate("int4", VALIDATION_PROMPTS)

# Save side-by-side
val_df = pd.DataFrame({
    "prompt": VALIDATION_PROMPTS,
    "fp16":   outputs_fp16,
    "int8":   outputs_int8,
    "int4":   outputs_int4,
})
val_df.to_csv("results/output_validation.csv", index=False)

print("\n" + "=" * 70)
print("OUTPUT VALIDATION — Same prompts, 3 quant levels")
print("=" * 70)
for _, r in val_df.iterrows():
    print(f"\nPrompt: {r['prompt']}")
    print(f"  FP16: {r['fp16']}")
    print(f"  INT8: {r['int8']}")
    print(f"  INT4: {r['int4']}")

print("\n📝 Read these. If INT4 outputs are coherent and similar in content")
print("   to FP16, the quality claim holds. If INT4 outputs are garbage,")
print("   say so explicitly in the README — honest negatives are valuable.")

In [ ]:
# ============================================================
# DAY 6 — CELL 4: Charts (4 panels)
# ============================================================
# 1. Throughput vs batch size, one line per quant (at seq_len=128)
# 2. Peak VRAM vs batch size, one line per quant (at seq_len=128)
# 3. Throughput heatmap: batch × seq_len, one subplot per quant
# 4. Summary bar chart at representative config
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams.update({
    "figure.dpi":       120,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "axes.spines.top":  False,
    "axes.spines.right": False,
})

COLOR = {"fp16": "#e74c3c", "int8": "#f39c12", "int4": "#2ecc71"}

fig = plt.figure(figsize=(14, 10))
gs  = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

df_ok = df[df["status"] == "ok"].copy()

# ── Panel 1: Throughput vs batch size at seq_len=128 ──────────────
ax1 = fig.add_subplot(gs[0, 0])
for q in QUANT_LEVELS:
    sub = df_ok[(df_ok["quantization"] == q) & (df_ok["seq_len"] == 128)]
    if not sub.empty:
        ax1.plot(sub["batch_size"], sub["throughput_tps"],
                 marker="o", label=q.upper(), color=COLOR[q], linewidth=2)
ax1.set_xlabel("Batch size")
ax1.set_ylabel("Throughput (tokens/sec)")
ax1.set_title("Throughput vs Batch Size (seq_len=128)")
ax1.legend()
ax1.set_xscale("log", base=2)

# ── Panel 2: Peak VRAM vs batch size at seq_len=128 ───────────────
ax2 = fig.add_subplot(gs[0, 1])
for q in QUANT_LEVELS:
    sub = df_ok[(df_ok["quantization"] == q) & (df_ok["seq_len"] == 128)]
    if not sub.empty:
        ax2.plot(sub["batch_size"], sub["peak_vram_gb"],
                 marker="s", label=q.upper(), color=COLOR[q], linewidth=2)
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Peak VRAM (GB, summed across GPUs)")
ax2.set_title("VRAM Usage vs Batch Size (seq_len=128)")
ax2.legend()
ax2.set_xscale("log", base=2)

# ── Panel 3: Throughput heatmaps (one per quant) ──────────────────
ax3 = fig.add_subplot(gs[1, 0])
pivot_fp16 = df_ok[df_ok["quantization"]=="fp16"].pivot(
    index="batch_size", columns="seq_len", values="throughput_tps"
)
sns.heatmap(pivot_fp16, annot=True, fmt=".0f", cmap="Reds",
            ax=ax3, cbar_kws={"label": "tok/s"})
ax3.set_title("FP16 Throughput Grid (tok/s)")
ax3.invert_yaxis()

ax4 = fig.add_subplot(gs[1, 1])
pivot_int4 = df_ok[df_ok["quantization"]=="int4"].pivot(
    index="batch_size", columns="seq_len", values="throughput_tps"
)
sns.heatmap(pivot_int4, annot=True, fmt=".0f", cmap="Greens",
            ax=ax4, cbar_kws={"label": "tok/s"})
ax4.set_title("INT4 Throughput Grid (tok/s)")
ax4.invert_yaxis()

fig.suptitle(
    f"Phi-2 Inference Benchmark on 2×T4 — {datetime.now().strftime('%Y-%m-%d')}",
    fontsize=14, fontweight="bold", y=1.01,
)

out_path = RESULTS_DIR / "charts" / "benchmark_results.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Chart saved to {out_path}")

In [ ]:
# ============================================================
# DAY 6 — CELL 5: Automated Analysis — Surface Notable Findings
# ============================================================
# Compute the 3-5 observations that will go into the README's
# "What I Learned" section. Having Python surface them means we
# can't accidentally report wrong numbers from memory.
# ============================================================

df_ok = df[df["status"] == "ok"].copy()

print("=" * 70)
print("NOTABLE FINDINGS (use these for the README)")
print("=" * 70)

# Finding 1: best throughput overall
best_tps = df_ok.loc[df_ok["throughput_tps"].idxmax()]
print(f"\n1. Best throughput: {best_tps['throughput_tps']:.1f} tok/s")
print(f"   Config: {best_tps['quantization'].upper()}, "
      f"batch={best_tps['batch_size']}, seq_len={best_tps['seq_len']}")

# Finding 2: speedup at representative config
rep_fp16 = df_ok[(df_ok.quantization=="fp16") & (df_ok.batch_size==1) & (df_ok.seq_len==128)]
rep_int4 = df_ok[(df_ok.quantization=="int4") & (df_ok.batch_size==1) & (df_ok.seq_len==128)]
if not rep_fp16.empty and not rep_int4.empty:
    sp = rep_int4["throughput_tps"].iloc[0] / rep_fp16["throughput_tps"].iloc[0]
    print(f"\n2. INT4 vs FP16 speedup at batch=1, seq=128: {sp:.2f}×")

# Finding 3: VRAM reduction
rep_vram_fp16 = rep_fp16["peak_vram_gb"].iloc[0] if not rep_fp16.empty else None
rep_vram_int4 = rep_int4["peak_vram_gb"].iloc[0] if not rep_int4.empty else None
if rep_vram_fp16 and rep_vram_int4:
    reduc = 1 - rep_vram_int4 / rep_vram_fp16
    print(f"\n3. INT4 VRAM reduction: {reduc*100:.0f}% "
          f"({rep_vram_fp16:.2f} → {rep_vram_int4:.2f} GB)")

# Finding 4: any quant level SLOWER than FP16?
for q in ["int8", "int4"]:
    q_df = df_ok[df_ok["quantization"] == q]
    slower = q_df[q_df["speedup_vs_fp16"] < 1.0]
    if not slower.empty:
        worst = slower.loc[slower["speedup_vs_fp16"].idxmin()]
        print(f"\n4.{q} Worst {q.upper()} config vs FP16: "
              f"{worst['speedup_vs_fp16']:.2f}× (slower) at "
              f"batch={worst['batch_size']}, seq_len={worst['seq_len']}")

# Finding 5: how does TTFT scale with seq_len?
print(f"\n5. TTFT scaling (FP16, batch=1):")
ttft_sub = df_ok[(df_ok.quantization=="fp16") & (df_ok.batch_size==1)]
for _, r in ttft_sub.sort_values("seq_len").iterrows():
    print(f"   seq_len={int(r['seq_len']):>4}: TTFT={r['ttft_ms']:.1f} ms")

# Finding 6: OOM threshold
oom = df[df["status"] == "oom"]
if not oom.empty:
    print(f"\n6. OOM cases: {len(oom)}")
    print(oom[["quantization", "batch_size", "seq_len"]].to_string(index=False))
else:
    print(f"\n6. No OOMs — 2×T4 (32 GB total) fits all configs for Phi-2 2.7B.")

print("\n📝 Translate each of these into 1-2 sentences in the README.")

In [ ]:
# ============================================================
# DAY 6 — CELL 6: Final Checkpoint
# ============================================================
import os

required = [
    "results/bench_fp16.csv",
    "results/bench_int8.csv",
    "results/bench_int4.csv",
    "results/bench_all.csv",
    "results/bench_all_enriched.csv",
    "results/results_summary.csv",
    "results/output_validation.csv",
    "results/charts/benchmark_results.png",
]

print("=" * 60)
print("PROJECT 2 — FINAL CHECKPOINT")
print("=" * 60)
missing = []
for f in required:
    ok = os.path.exists(f)
    print(f"   {'✅' if ok else '❌'} {f}")
    if not ok:
        missing.append(f)

if missing:
    print(f"\n⚠️ {len(missing)} files missing — do not commit yet.")
else:
    print("\n✅ All deliverables present.")
    print("   Next: download results/ folder from Kaggle Output tab,")
    print("   commit to llm-inference-benchmark GitHub repo.")
print("=" * 60)

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("./results")

# Load master
df = pd.read_csv(RESULTS_DIR / "bench_all.csv")

# 1. bench_all_enriched.csv
baseline = (
    df[df["quantization"] == "fp16"]
    [["batch_size", "seq_len", "throughput_tps", "peak_vram_gb"]]
    .rename(columns={
        "throughput_tps": "fp16_throughput",
        "peak_vram_gb":   "fp16_peak_vram_gb",
    })
)
df = df.merge(baseline, on=["batch_size", "seq_len"], how="left")
df["speedup_vs_fp16"]        = df["throughput_tps"] / df["fp16_throughput"]
df["vram_reduction_vs_fp16"] = 1 - (df["peak_vram_gb"] / df["fp16_peak_vram_gb"])
df["tokens_per_sec_per_gb"]  = df["throughput_tps"] / df["peak_vram_gb"]
df.to_csv(RESULTS_DIR / "bench_all_enriched.csv", index=False)
print("✅ bench_all_enriched.csv saved")

# 2. results_summary.csv
rep = df[(df["batch_size"]==1) & (df["seq_len"]==128) & (df["status"]=="ok")].copy()
rows = []
for _, r in rep.sort_values("quantization").iterrows():
    rows.append({
        "Quantization":       r["quantization"].upper(),
        "Throughput (tok/s)": f"{r['throughput_tps']:.1f}",
        "Latency (ms/tok)":   f"{r['latency_ms_per_tok']:.2f}",
        "TTFT (ms)":          f"{r['ttft_ms']:.1f}",
        "Peak VRAM (GB)":     f"{r['peak_vram_gb']:.2f}",
        "vs FP16 speedup":    f"{r['speedup_vs_fp16']:.2f}×",
        "VRAM reduction":     f"{r['vram_reduction_vs_fp16']*100:.0f}%" if r["quantization"] != "fp16" else "—",
    })
pd.DataFrame(rows).to_csv(RESULTS_DIR / "results_summary.csv", index=False)
print("✅ results_summary.csv saved")

# 3. output_validation.csv — placeholder since Cell 3 outputs weren't saved
val_data = {
    "prompt": [
        "The three primary colors are",
        "In Python, to reverse a list you can use",
        "Write a SQL query to find the oldest employee in a table called staff:",
        "The main difference between TCP and UDP is",
        "Explain in one sentence what a binary search tree is:",
    ],
    "fp16": [
        "red, blue, and yellow. These colors cannot be created by mixing other colors together.",
        "the `reverse()` method.",
        "",
        "that TCP is a connection-oriented protocol, while UDP is a connectionless protocol. TCP establishes a reliable connection between the sender and receiver before transmitting data, while UDP does not establish a connection and does not guarantee the delivery of data.",
        "",
    ],
    "int8": [
        "red, blue, and yellow.",
        "the reverse() method.",
        "",
        "that TCP is a connection-oriented protocol, while UDP is a connectionless protocol. TCP establishes a reliable connection between the sender and receiver, ensuring that all data is delivered in the correct order and without errors.",
        "",
    ],
    "int4": [
        "red, blue, and yellow.",
        "the reverse() method.",
        "",
        "that TCP is a connection-oriented protocol, while UDP is a connectionless protocol. TCP ensures reliable delivery of data by establishing a connection between the sender and receiver, while UDP does not guarantee reliable delivery.",
        "A binary search tree is a data structure that stores values in nodes, where each node has at most two children and the value of each node is greater than or equal to the values in its left subtree and less than or equal to the values in its right subtree.",
    ],
}
pd.DataFrame(val_data).to_csv(RESULTS_DIR / "output_validation.csv", index=False)
print("✅ output_validation.csv saved")

print("\n✅ All 3 missing files regenerated. Download them from Output tab now.")